In [1]:
import polars as pl
import pandas as pd
import glob
import re
import os
import yaml
import pysam


In [12]:
config_path = "/user/leuven/350/vsc35059/osiga/ASAP/asap_git/wasp_pipeline/test/config.yaml"
config = yaml.safe_load(open(config_path))

analysis_dir = config["main"]["analysis_dir"]
print(analysis_dir) 

bcf_path = config["data"]["bcf_combined"]
bcf = pysam.VariantFile(bcf_path)

# get contig names and samples from the bcf 
chroms = [x for x in bcf.header.contigs]
samples = [x for x in bcf.header.samples]

# makeoutput directory
outdir = analysis_dir + "/phasing_blocks"
os.makedirs(outdir, exist_ok=True)

/staging/leuven/stg_00002/lcb/osiga/ASAP/analysis/wasp_indel_test_as_counts


In [21]:
# parse BCF file and extract phasing information for each sample


for chrom in chroms:

    print(chrom)
    phasing_dict = {}
    i = 0

    # extract phasing info for all samples per chromosome
    for record in bcf.fetch(contig=chrom):

        for sample in record.samples:

            try:
                phasing_block = record.samples[sample]['PS']
                if phasing_block:

                    l = [record.chrom, record.pos, sample, record.samples[sample]['PS']]
                    if sample in phasing_dict:
                        phasing_dict[sample].append(l)
                    else:
                        phasing_dict[sample] = [l]
            except:
                # skip unphased variants
                continue
            
        if i % 200000 == 0:
            print(i)
        i += 1

    # write phasing blocks to csv files for each sample
    for sample in samples:

        # convert list of lists to dataframe
        try:
            df = pd.DataFrame(phasing_dict[sample], columns=['chrom', 'pos', 'sample', 'phasing_block'])
            df = pl.from_pandas(df)

            (df
                .select("chrom", "pos", "phasing_block")
                .group_by(["chrom", "phasing_block"])
                .agg(pl.col("pos").min().alias("start"), 
                    pl.col("pos").max().alias("end"))
                .select(["chrom", "start", "end", "phasing_block"])
                .sort("chrom", "start")
                .write_csv(f"{outdir}/pb_{sample}_{chrom}.bed", separator = "\t", include_header = False)
            )
        except:
            #print(f"Sample {sample} not found in phasing_dict")
            continue

chr1
0
200000
400000
chr2
0
200000
400000
chr3
0
200000
chr4
0
200000
400000
chr5
0
200000
chr6
0
200000
chr7
0
200000
chr8
0
200000
chr9
0
200000
400000
chr10
0
200000
chr11
0
200000
chr12
0
200000
chr13
0
200000
chr14
0
200000
chr15
0
200000
chr16
0
200000
chr17
0
chr18
0
chr19
0
chr20
0
chr21
0
chr22
0
chrX
0
200000
chrY
0
chrM
